# Train Coco's voice on Kaggle (Piper, high quality)

Fine-tunes Piper's **en_US lessac high** voice on Coco's dataset. Built to run **headless**: start it
with **Save Version > Save & Run All (Commit)** and close the browser. Each run trains for
`TRAIN_HOURS`, stops cleanly at the end of an epoch, and leaves its results in the run's **Output**:

- `coco-epochNNNN.ckpt`: where training got to (the next run continues from it)
- `export/en_US-coco-high.onnx` + `.onnx.json`: the voice so far, ready for the repo's `models/piper/`
- `listen/*.wav`: four lines Coco would say, to judge it by ear
- `train.log`

**One-time setup**
1. Make a Kaggle dataset called `coco-voice` with `coco-voice-dataset.zip` in it (Kaggle may unzip it:
   either way works). To continue the Colab run, also put its checkpoint (`coco-epochNNNN.ckpt` from
   `My Drive/coco-voice/checkpoint/`) in that dataset.
2. In this notebook: **Add Input** > the `coco-voice` dataset.
3. Settings (right panel): **Accelerator: GPU T4** (one GPU is used), **Internet: on** (needs a
   phone-verified account; it downloads Piper and, on a first run without a checkpoint, the lessac voice).

**Every run after the first:** Add Input > **Your Work** > this notebook's output (the newest version).
The notebook takes the checkpoint with the **highest epoch** from everything attached, and prints it at
the start, so check that line. Then Save & Run All again.

**When is it done?** Listen to `listen/` after each run. Stop when it sounds right (usually a few
hundred epochs past the start). More is not always better: pick by ear.

In [ ]:
# ---- Settings ----
TRAIN_HOURS = 8.0         # Training time per run. Kaggle's GPU runs are capped (reports say 9-12 h, check
                          # your account): this leaves room for setup and saving. A run that hits the cap
                          # loses its whole output, so don't go above the cap minus ~45 minutes.
HARD_STOP_MINUTES = 25    # After TRAIN_HOURS, wait at most this long for the current epoch's checkpoint
EXTRA_EPOCHS = 1000       # Upper limit on epochs past the lessac start (time is the real limit per run)
BATCH_SIZE = 16           # Lower to 8 if you get "CUDA out of memory"
PRECISION = "16-mixed"    # About 2x faster on a T4. If losses show "nan", change to "32" (it resumes)
PIPER_COMMIT = "39ac0a04ff01120b1cb7867d21723fa320305b03"  # piper1-gpl version this notebook was written for
VOICE_NAME = "coco"
LESSAC_EPOCH = 2218

# Same working paths as the Colab notebook (/content/...), so checkpoints move between the two cleanly.
# Only /kaggle/working is kept after a run: everything else is scratch.
import glob, os, sys, zipfile, importlib
WORK = "/kaggle/working"
DATA, CACHE, RUNS, PIPER = "/content/data", "/content/cache", "/content/runs", "/content/piper1-gpl"
SCRATCH = "/content/scratch"
LOG = f"{SCRATCH}/train.log"
CONFIG_JSON = f"{SCRATCH}/config.json"
for d in (DATA, CACHE, RUNS, SCRATCH):
    os.makedirs(d, exist_ok=True)

def ckpt_epoch(path):
    import torch
    return torch.load(path, map_location="cpu", weights_only=True, mmap=True)["epoch"]

def best_checkpoint(paths):
    """The complete checkpoint with the highest epoch (input files' dates mean nothing on Kaggle)."""
    best = None
    for p in paths:
        if "lessac" in os.path.basename(p).lower() or not zipfile.is_zipfile(p):
            continue
        try:
            key = (ckpt_epoch(p), os.path.getmtime(p))
        except Exception as e:
            print("skipping unreadable checkpoint", p, e)
            continue
        if best is None or key > best[0]:
            best = (key, p)
    return best[1] if best else None

def use_piper():
    """Lets this notebook's Python import the editable Piper install (it started before the install)."""
    src = f"{PIPER}/src"
    if not os.path.isdir(f"{src}/piper"):
        raise RuntimeError("Piper isn't installed yet: run the Install cell first.")
    sys.path_importer_cache.pop(src, None)
    if src not in sys.path:
        sys.path.insert(0, src)
    importlib.invalidate_caches()

In [ ]:
# ---- 1. GPU, internet, and Piper training (pinned version) ----
# First, before anything imports torch: Piper's install may change the torch version.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("No internet: turn it on in Settings (right panel) > Internet") from e
!apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
!test -d {PIPER} || git clone -q https://github.com/OHF-Voice/piper1-gpl.git {PIPER}
%cd {PIPER}
!git checkout -q {PIPER_COMMIT}
!pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
# The in-place build of the eSpeak bridge runs setup.py directly, so its build tools must be in this Python
!pip -q install scikit-build setuptools wheel cmake ninja
!python3 setup.py build_ext --inplace > /tmp/build_ext.log 2>&1 || (tail -30 /tmp/build_ext.log; false)
!python3 -c "from piper import espeakbridge; from piper.train.vits import monotonic_align; print('eSpeak bridge and monotonic_align OK')"
use_piper()

In [ ]:
# ---- 2. Dataset and the checkpoint to continue from ----
import shutil, subprocess
meta = sorted(glob.glob("/kaggle/input/**/metadata.csv", recursive=True))
zips = sorted(glob.glob("/kaggle/input/**/coco-voice-dataset*.zip", recursive=True))
if not os.path.exists(f"{DATA}/metadata.csv"):
    if meta:   # Kaggle unzipped it on upload
        src = os.path.dirname(meta[0])
        shutil.copy(f"{src}/metadata.csv", DATA)
        shutil.copytree(f"{src}/wavs", f"{DATA}/wavs", dirs_exist_ok=True)
    elif zips:
        subprocess.run(["unzip", "-q", "-o", zips[0], "-d", DATA], check=True)
    else:
        raise FileNotFoundError("No dataset: Add Input > the coco-voice dataset (with coco-voice-dataset.zip)")
print("clips:", sum(1 for _ in open(f"{DATA}/metadata.csv", encoding="utf-8")))

RESUME = best_checkpoint(glob.glob("/kaggle/input/**/*.ckpt", recursive=True))
if RESUME:
    print(f"CONTINUING FROM EPOCH {ckpt_epoch(RESUME)} ({ckpt_epoch(RESUME) - LESSAC_EPOCH} past the lessac start): {RESUME}")
else:
    print("No checkpoint attached: starting from the lessac high voice. (If you meant to continue, add the "
          "previous run's output as an input and run again.)")

In [ ]:
# ---- 3. The "high" model shape; the lessac checkpoint only when there's nothing to continue ----
# (The old trainer's --quality high; the new trainer's defaults are medium. "1" is quoted on purpose:
#  the model compares resblock == "1", and an unquoted 1 would quietly build the wrong vocoder.)
HIGH_YAML = """
model:
  resblock: "1"
  resblock_kernel_sizes: [3, 7, 11]
  resblock_dilation_sizes: [[1, 3, 5], [1, 3, 5], [1, 3, 5]]
  upsample_rates: [8, 8, 2, 2]
  upsample_initial_channel: 512
  upsample_kernel_sizes: [16, 16, 4, 4]
"""
open(f"{SCRATCH}/high.yaml", "w").write(HIGH_YAML)

LESSAC_CLEAN = None
if not RESUME:
    use_piper()
    import inspect, pathlib, torch, yaml
    from piper.train.vits.lightning import VitsModel
    LESSAC = f"{SCRATCH}/lessac-high.ckpt"
    !test -f {LESSAC} || wget -q -O {LESSAC} "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/high/epoch%3D2218-step%3D838782.ckpt?download=true"
    ck = torch.load(LESSAC, map_location="cpu", weights_only=False)  # official rhasspy checkpoint (old format)
    own = VitsModel(**yaml.safe_load(HIGH_YAML)["model"], mos_metric=None).state_dict()
    fits = [k for k, v in ck["state_dict"].items() if k in own and own[k].shape == v.shape]
    print(f"{len(fits)} of {len(ck['state_dict'])} checkpoint weights fit the high model")
    assert len(fits) == len(ck["state_dict"]), "Shape mismatch: don't train, the model settings are wrong"
    # Lightning resumes with the safe loader (weights_only=True): paths become plain strings, and only the
    # settings the new trainer accepts are kept (it re-applies them as --model.* options on resume).
    def plain(o):
        if isinstance(o, pathlib.PurePath): return str(o)
        if isinstance(o, dict): return {k: plain(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return type(o)(plain(v) for v in o)
        return o
    ck = {k: v if k == "state_dict" else plain(v) for k, v in ck.items()}
    accepted = set(inspect.signature(VitsModel.__init__).parameters) - {"self", "kwargs", "dataset", "batch_size", "num_symbols"}
    ck["hyper_parameters"] = {k: v for k, v in ck.get("hyper_parameters", {}).items() if k in accepted}
    LESSAC_CLEAN = f"{SCRATCH}/lessac-high-clean.ckpt"
    torch.save(ck, LESSAC_CLEAN)
    del ck, own
    torch.load(LESSAC_CLEAN, map_location="cpu", weights_only=True)
    print("Cleaned lessac checkpoint passes the safe loader")

In [ ]:
# ---- 4. Train for TRAIN_HOURS, then stop cleanly right after an epoch's checkpoint is saved ----
import signal, time
# Keep 3 checkpoints, not 11: Piper always adds its own savers (5 best by val_mel, 5 best by val_mos,
# last.ckpt). Both "5 best" become 1.
MAIN = f"{PIPER}/src/piper/train/__main__.py"
src = open(MAIN).read()
if src.count("save_top_k=5") == 2:
    open(MAIN, "w").write(src.replace("save_top_k=5", "save_top_k=1"))
elif "save_top_k=5" in src or src.count("save_top_k=1") < 3:
    raise RuntimeError("Piper's checkpoint settings aren't where this notebook expects; nothing changed.")

start = f"--ckpt_path '{RESUME or LESSAC_CLEAN}'"
cmd = (f"python3 -m piper.train fit --config {SCRATCH}/high.yaml "
       f"--data.voice_name {VOICE_NAME} --data.csv_path {DATA}/metadata.csv "
       f"--data.audio_dir {DATA}/wavs --model.sample_rate 22050 --data.espeak_voice en-us "
       f"--data.cache_dir {CACHE} --data.config_path {CONFIG_JSON} "
       f"--data.batch_size {BATCH_SIZE} --data.length_bucketing true "
       f"--trainer.accelerator gpu --trainer.devices 1 --trainer.precision {PRECISION} "
       f"--trainer.max_epochs {LESSAC_EPOCH + EXTRA_EPOCHS} --trainer.default_root_dir {RUNS} {start}")
print(cmd)
log = open(LOG, "a")
proc = subprocess.Popen(cmd, shell=True, cwd=PIPER, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

def run_checkpoints():
    return glob.glob(f"{RUNS}/**/*.ckpt", recursive=True)

def last_line():
    try:
        tail = open(LOG, "rb").read()[-3000:].decode("utf-8", "replace").replace("\r", "\n")
        lines = [l for l in tail.splitlines() if l.strip()]
        return lines[-1][-160:] if lines else ""
    except OSError:
        return ""

t0 = time.time()
soft_stop, hard_stop = t0 + TRAIN_HOURS * 3600, t0 + TRAIN_HOURS * 3600 + HARD_STOP_MINUTES * 60
last_report, waiting_since = 0, None
while proc.poll() is None:
    time.sleep(20)
    now = time.time()
    if now - last_report > 900:   # Every 15 minutes, into the run's log
        last_report = now
        newest = best_checkpoint(run_checkpoints())
        print(f"[{(now - t0) / 3600:.2f} h] newest epoch: {ckpt_epoch(newest) if newest else '-'} | {last_line()}", flush=True)
    if now >= soft_stop:
        # Wait for the next epoch's last.ckpt to be completely written (a stop mid-save leaves a broken file)
        if waiting_since is None:
            waiting_since = now
            print("Time's up: stopping after the current epoch is saved...", flush=True)
        fresh = [p for p in run_checkpoints() if os.path.basename(p) == "last.ckpt" and os.path.getmtime(p) > waiting_since]
        if fresh and now - os.path.getmtime(fresh[0]) > 30 and zipfile.is_zipfile(fresh[0]):
            break
        if now >= hard_stop:
            print("The epoch didn't finish in time: stopping anyway (the previous epoch's checkpoint is kept).")
            break
if proc.poll() is None:
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
log.close()
print(f"Training ended after {(time.time() - t0) / 3600:.2f} h, exit code {proc.returncode}. Last log line: {last_line()}")

In [ ]:
# ---- 5. Save the results to the Output: checkpoint, voice, samples, log ----
use_piper()
import wave
for old in glob.glob(f"{WORK}/*"):   # Only this run's results in the Output
    shutil.rmtree(old) if os.path.isdir(old) else os.remove(old)
final = best_checkpoint(run_checkpoints())
if final is None and RESUME:
    print("This run saved no checkpoint (see train.log). Passing on the one it started from, so the next run can continue.")
    final = RESUME
shutil.copy(LOG, f"{WORK}/train.log")
assert final, "No checkpoint at all: see train.log in the Output"
epoch = ckpt_epoch(final)
shutil.copy(final, f"{WORK}/coco-epoch{epoch}.ckpt")
print(f"Saved coco-epoch{epoch}.ckpt ({epoch - LESSAC_EPOCH} epochs past the lessac start)")

os.makedirs(f"{WORK}/export", exist_ok=True)
onnx = f"{WORK}/export/en_US-{VOICE_NAME}-high.onnx"
!python3 -m piper.train.export_onnx --checkpoint "{final}" --output-file "{onnx}"
shutil.copy(CONFIG_JSON, onnx + ".json")

from piper import PiperVoice
voice = PiperVoice.load(onnx, config_path=onnx + ".json")
os.makedirs(f"{WORK}/listen", exist_ok=True)
for i, line in enumerate([
    "Hey. What are we fixing today?",
    "Your shopping list has eggs, oat milk, flour and coffee beans. Butter wasn't on it, so I added it.",
    "Black jeans and a white tee? Clean. The boots are doing the heavy lifting, babe.",
    "Okay. Breathe. Your notes are all still here. Nothing's gone.",
]):
    with wave.open(f"{WORK}/listen/epoch{epoch}_{i}.wav", "wb") as wf:
        voice.synthesize_wav(line, wf)
!ls -la {WORK} {WORK}/export {WORK}/listen

In [ ]:
# ---- 6. Listen here (only useful when running interactively; a committed run saves the files above) ----
from IPython.display import Audio, display
for path in sorted(glob.glob(f"{WORK}/listen/*.wav")):
    print(os.path.basename(path)); display(Audio(path))